In [ ]:
# Cell 1 - Compute mean and std at 128x128
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

dataset = datasets.ImageFolder(root='lavender_dataset_splitted_stress', transform=transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
]))

loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0)
mean = 0.0
std = 0.0
total_images = 0

for images, _ in loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_images += batch_samples

mean /= total_images
std /= total_images

print(f"Mean: {mean}")
print(f"Std: {std}")

In [ ]:
# Cell 2 - Model + training functions
import os
import argparse

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

class SimpleCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class TransformWrapper:
    """Wraps a dataset subset to apply different transforms at runtime"""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    
    def __len__(self):
        return len(self.subset)
    
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


def plot_loss_curves(train_losses, val_losses, fold_idx):
    plt.figure(figsize=(10, 5))
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, 'b-o', label='Training Loss', linewidth=2, markersize=6)
    plt.plot(epochs, val_losses, 'r-o', label='Validation Loss', linewidth=2, markersize=6)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title(f'Training vs Validation Loss - Fold {fold_idx + 1}', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def get_loaders_kfold(data_dir, batch_size=32, img_size=256, n_splits=5, seed=42, mean=None, std=None):
    # Load dataset ONCE without transforms (raw PIL images)
    full_dataset = datasets.ImageFolder(root=data_dir, transform=None)
    num_classes = len(full_dataset.classes)
    dataset_size = len(full_dataset)
    
    labels = full_dataset.targets
    
    # Define transforms
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean.tolist(), std=std.tolist()),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean.tolist(), std=std.tolist()),
    ])

    # Create stratified folds
    kfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(kfold.split(range(dataset_size), labels))

    fold_loaders = []
    for train_idx, val_idx in splits:
        # Create subsets from the SAME dataset
        train_subset = Subset(full_dataset, train_idx)
        val_subset = Subset(full_dataset, val_idx)
        
        # Wrap subsets with different transforms
        train_wrapped = TransformWrapper(train_subset, train_transform)
        val_wrapped = TransformWrapper(val_subset, val_transform)
        
        train_loader = DataLoader(train_wrapped, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
        val_loader = DataLoader(val_wrapped, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

        fold_loaders.append((train_loader, val_loader))

    return fold_loaders, num_classes


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [ ]:
# Cell 3 - Main training loop
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-dir', type=str, default='lavender_dataset_splitted_stress')
    parser.add_argument('--epochs', type=int, default=25)            # ← changed from 10
    parser.add_argument('--batch-size', type=int, default=32)
    parser.add_argument('--img-size', type=int, default=128)
    parser.add_argument('--lr', type=float, default=1e-4)
    parser.add_argument('--weight-decay', type=float, default=1e-3)  # ← new
    parser.add_argument('--save-path', type=str, default='./models_weights')
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
    parser.add_argument('--n-splits', type=int, default=5)
    args = parser.parse_args([])

    if not os.path.isdir(args.data_dir):
        raise SystemExit(f"Dataset directory not found: {args.data_dir}")

    fold_loaders, num_classes = get_loaders_kfold(
        args.data_dir,
        batch_size=args.batch_size,
        img_size=args.img_size,
        n_splits=args.n_splits,
        mean=mean,
        std=std
    )

    device = torch.device(args.device)
    criterion = nn.CrossEntropyLoss()
    fold_results = []

    for fold_idx, (train_loader, val_loader) in enumerate(fold_loaders):
        print(f"\n{'='*60}")
        print(f"Fold {fold_idx + 1}/{args.n_splits}")
        print(f"Train size: {len(train_loader.dataset)}, Val size: {len(val_loader.dataset)}")
        print(f"{'='*60}")

        model = SimpleCNN(in_channels=3, num_classes=num_classes).to(device)
        
        # ← weight_decay added here
        optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        
        # ← scheduler added here — watches val_loss, patience=5, halves LR each time
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',        # we're monitoring val_loss (lower = better)
            patience=5,        # wait 5 epochs of no improvement before reducing
            factor=0.4,        # multiply LR by 0.4 when triggered
            verbose=True       # prints a message when LR is reduced
        )

        best_val_loss = float('inf')   # ← track best val_loss instead of val_acc
        best_val_acc = 0.0
        train_losses = []
        val_losses = []

        for epoch in range(1, args.epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc = validate(model, val_loader, criterion, device)

            train_losses.append(train_loss)
            val_losses.append(val_loss)

            # ← step the scheduler with val_loss
            scheduler.step(val_loss)

            print(f"Epoch {epoch}/{args.epochs}: "
                  f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

            # ← save on best val_loss instead of val_acc (more stable with small data)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_val_acc = val_acc
                save_path = f"{args.save_path}_fold{fold_idx + 1}.pth"
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'epoch': epoch,
                    'num_classes': num_classes,
                    'fold': fold_idx + 1,
                    'val_loss': val_loss,
                    'val_acc': val_acc,
                }, save_path)
                print(f"Saved improved model to {save_path} "
                      f"(val_loss={val_loss:.4f}, val_acc={val_acc:.4f})")

        plot_loss_curves(train_losses, val_losses, fold_idx)
        fold_results.append(best_val_acc)
        print(f"Fold {fold_idx + 1} Best Val Acc: {best_val_acc:.4f}\n")

    print(f"\n{'='*60}")
    print("K-Fold Cross-Validation Summary")
    print(f"{'='*60}")
    for fold_idx, acc in enumerate(fold_results):
        print(f"Fold {fold_idx + 1}: {acc:.4f}")
    print(f"Mean Accuracy: {sum(fold_results)/len(fold_results):.4f}")
    print(f"Std Dev: {(sum((x - sum(fold_results)/len(fold_results))**2 for x in fold_results) / len(fold_results))**0.5:.4f}")
    print(f"{'='*60}")


main()